# Multivariate Time Series Workflow with chronobox

This notebook demonstrates a complete end-to-end pipeline for multivariate time series analysis
using the **chronobox** library. We work with US macroeconomic quarterly data (GDP growth,
inflation, federal funds rate, unemployment) and cover:

1. Exploratory analysis and cross-correlations
2. Unit root tests for each variable
3. VAR lag selection and Johansen cointegration test
4. Decision tree: VAR in differences vs VECM vs ARDL
5. Model estimation
6. Impulse Response Functions (IRF) and FEVD
7. Granger causality
8. Structural VAR (SVAR) with Cholesky decomposition
9. Spillover index (Diebold-Yilmaz)
10. Multivariate forecasting

**Dataset**: `us_macro_quarterly.csv` - US GDP growth, inflation, fed funds rate, unemployment (1975-2024)

## Pipeline Flowchart

```
Multivariate Data (K variables)
   |
   v
[1. Exploratory Analysis] ---> Plots, correlations, summary statistics
   |
   v
[2. Unit Root Tests] -------> ADF/KPSS for each variable
   |                            |
   |                     All I(0)? ---> Yes ---> VAR in levels
   |                            |
   |                     All I(1)?
   |                       /        \
   |                      v          v
[3. Cointegration] --> Johansen Test     ARDL Bounds Test
   |                    |                     |
   |              Cointegrated?          Long-run relation?
   |              /          \           /             \
   |             v            v         v               v
   |           VECM     VAR in diff   ARDL-ECM    VAR in diff
   |
   v
[4. Estimation] ---------> Fit chosen model
   |
   v
[5. Analysis] -----------> IRF (impulse responses)
   |                       FEVD (variance decomposition)
   |                       Granger causality
   |
   v
[6. Structural Analysis] -> SVAR (Cholesky ordering)
   |                        Structural IRF
   |
   v
[7. Spillover] -----------> Diebold-Yilmaz spillover index
   |                        Network graph
   |
   v
[8. Forecast] ------------> h-step-ahead multivariate forecast
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# chronobox imports - Models
from chronobox import VAR, VECM, SVAR, BayesianVAR
from chronobox.models import ARDL

# chronobox imports - Analysis
from chronobox.analysis import IRF, FEVD, SpilloverIndex, granger_causality

# chronobox imports - Tests
from chronobox.tests_stat import adf_test, kpss_test, ljung_box_test

# chronobox imports - Visualization
from chronobox.visualization import (
    plot_series, plot_irf, plot_fevd, plot_forecast,
    plot_heatmap, plot_network, set_theme
)

set_theme('professional')

print('chronobox imported successfully')

---
## 1. Data Loading and Exploratory Analysis

In [ ]:
# Load US macro data
import os
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
df = pd.read_csv(os.path.join(data_dir, 'us_macro_quarterly.csv'), parse_dates=['date'])
df = df.set_index('date')
df.index.freq = 'QS'

var_names = ['gdp', 'inflation', 'fed_funds', 'unemployment']

print(f'Period: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Observations: {len(df)}')
print(f'Variables: {list(df.columns)}')
print()
df.describe().round(4)

In [ ]:
# GRAPH 1: Plot all variables
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

titles = ['GDP Growth (%)', 'Inflation (%)', 'Federal Funds Rate (%)', 'Unemployment (%)']
colors = ['steelblue', 'darkorange', 'green', 'red']

for ax, col, title, color in zip(axes.flat, var_names, titles, colors):
    ax.plot(df.index, df[col], color=color, linewidth=1.2)
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=df[col].mean(), color='gray', linestyle='--', alpha=0.5)

plt.suptitle('US Macroeconomic Variables (Quarterly)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# GRAPH 2: Correlation matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation heatmap
corr = df[var_names].corr()
im = axes[0].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[0].set_xticks(range(len(var_names)))
axes[0].set_yticks(range(len(var_names)))
axes[0].set_xticklabels(var_names, rotation=45, ha='right')
axes[0].set_yticklabels(var_names)
for i in range(len(var_names)):
    for j in range(len(var_names)):
        axes[0].text(j, i, f'{corr.iloc[i, j]:.2f}',
                     ha='center', va='center', fontsize=10,
                     color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')
plt.colorbar(im, ax=axes[0], shrink=0.8)
axes[0].set_title('Correlation Matrix', fontsize=12)

# Scatter: GDP vs Unemployment (Phillips curve intuition)
axes[1].scatter(df['unemployment'], df['gdp'], alpha=0.5, c='steelblue', s=20)
z = np.polyfit(df['unemployment'], df['gdp'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['unemployment'].min(), df['unemployment'].max(), 100)
axes[1].plot(x_line, p(x_line), 'r--', linewidth=1.5)
axes[1].set_xlabel('Unemployment (%)')
axes[1].set_ylabel('GDP Growth (%)')
axes[1].set_title('GDP Growth vs Unemployment', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Unit Root Tests

For multivariate analysis, we need to determine the integration order of each variable.
This is crucial for choosing between VAR in levels, VAR in differences, or VECM.

In [ ]:
# Test each variable for unit root
print('=' * 70)
print(f'{"Variable":<15} {"ADF stat":>10} {"ADF p":>8} {"KPSS stat":>10} {"KPSS p":>8} {"Order":>6}')
print('=' * 70)

integration_orders = {}

for col in var_names:
    y_col = df[col].values
    
    # Level tests
    adf = adf_test(y_col, regression='c')
    kpss = kpss_test(y_col, regression='c')
    
    # Determine integration order
    if adf.reject and not kpss.reject:
        order = 'I(0)'
    else:
        # Test first difference
        dy = np.diff(y_col)
        adf_d = adf_test(dy, regression='c')
        if adf_d.reject:
            order = 'I(1)'
        else:
            order = 'I(2)?'
    
    integration_orders[col] = order
    print(f'{col:<15} {adf.test_statistic:>10.4f} {adf.pvalue:>8.4f} '
          f'{kpss.test_statistic:>10.4f} {kpss.pvalue:>8.4f} {order:>6}')

print('=' * 70)
print(f'\nIntegration orders: {integration_orders}')

In [ ]:
# Prepare data matrix
data = df[var_names].values
T, K = data.shape
print(f'Data matrix: T={T} observations, K={K} variables')
print(f'Variables: {var_names}')

---
## 3. VAR Lag Selection and Cointegration Test

In [ ]:
# Lag selection using information criteria
print('=== VAR Lag Selection ===')
print(f'{"Lags":>5} {"AIC":>12} {"BIC":>12} {"HQ":>12}')
print('-' * 45)

ic_results = {}
for p in range(1, 9):
    var_temp = VAR(lags=p, trend='c')
    res_temp = var_temp.fit(data, names=var_names)
    ic_results[p] = {'aic': res_temp.aic, 'bic': res_temp.bic, 'hqic': res_temp.hqic}
    print(f'{p:>5} {res_temp.aic:>12.4f} {res_temp.bic:>12.4f} {res_temp.hqic:>12.4f}')

best_aic = min(ic_results, key=lambda x: ic_results[x]['aic'])
best_bic = min(ic_results, key=lambda x: ic_results[x]['bic'])
best_hq = min(ic_results, key=lambda x: ic_results[x]['hqic'])

print(f'\nOptimal lags - AIC: {best_aic}, BIC: {best_bic}, HQ: {best_hq}')
p_opt = best_aic
print(f'Selected lag order: p = {p_opt} (AIC criterion)')

In [ ]:
# Johansen cointegration test
vecm_test = VECM(lags=p_opt, deterministic='ci')
johansen = vecm_test.johansen_test(data)

print('=== Johansen Cointegration Test ===')
print(f'\nTrace Test:')
print(f'{"H0: rank <= r":>15} {"Trace Stat":>12} {"95% CV":>10} {"Decision":>15}')
print('-' * 55)
for r in range(K):
    stat = johansen.trace_stat[r]
    cv = johansen.trace_crit[r, 1]  # 95% critical value
    reject = stat > cv
    print(f'{"r <= " + str(r):>15} {stat:>12.4f} {cv:>10.4f} {"Reject" if reject else "Fail to reject":>15}')

print(f'\nMax Eigenvalue Test:')
print(f'{"H0: rank = r":>15} {"Max-Eig Stat":>12} {"95% CV":>10} {"Decision":>15}')
print('-' * 55)
for r in range(K):
    stat = johansen.max_eig_stat[r]
    cv = johansen.max_eig_crit[r, 1]
    reject = stat > cv
    print(f'{"r = " + str(r):>15} {stat:>12.4f} {cv:>10.4f} {"Reject" if reject else "Fail to reject":>15}')

print(f'\nCointegration rank (trace):     {johansen.rank_trace}')
print(f'Cointegration rank (max-eigen):  {johansen.rank_maxeig}')

---
## 4. Decision Tree: VAR vs VECM vs ARDL

The choice of model depends on the properties of the data:

```
Are all variables I(0)?
  |-- Yes --> VAR in levels
  |-- No  --> Are all variables I(1)?
                |-- No  --> Mixed orders? --> ARDL bounds test
                |-- Yes --> Are they cointegrated?
                              |-- Yes --> VECM (preserves long-run)
                              |-- No  --> VAR in first differences
```

Let's implement this decision logic programmatically.

In [ ]:
# Automated decision tree
orders = list(integration_orders.values())
n_i0 = sum(1 for o in orders if o == 'I(0)')
n_i1 = sum(1 for o in orders if o == 'I(1)')

print('=== Model Selection Decision Tree ===')
print(f'Variables: {var_names}')
print(f'Integration orders: {orders}')
print(f'  I(0): {n_i0}, I(1): {n_i1}')

coint_rank = johansen.rank_trace

if n_i0 == K:
    decision = 'VAR_LEVELS'
    print(f'\n=> All variables are I(0)')
    print(f'=> Decision: VAR in LEVELS')
elif n_i1 == K:
    if coint_rank > 0:
        decision = 'VECM'
        print(f'\n=> All variables are I(1) with {coint_rank} cointegrating relation(s)')
        print(f'=> Decision: VECM with rank = {coint_rank}')
    else:
        decision = 'VAR_DIFF'
        print(f'\n=> All variables are I(1) but no cointegration found')
        print(f'=> Decision: VAR in FIRST DIFFERENCES')
else:
    decision = 'ARDL'
    print(f'\n=> Mixed integration orders')
    print(f'=> Decision: ARDL bounds test approach')

# We'll estimate all three models for pedagogical purposes
print(f'\n(For learning purposes, we will estimate VAR, VECM, and ARDL below)')

---
## 5. Model Estimation

### 5a. VAR Estimation

In [ ]:
# Fit VAR model
var_model = VAR(lags=p_opt, trend='c')
var_result = var_model.fit(data, names=var_names)

print('=== VAR Model Summary ===')
print(f'Lag order:   {var_result.k_ar}')
print(f'Variables:   {var_result.names}')
print(f'Obs (used):  {var_result.nobs}')
print(f'Stable:      {var_result.is_stable}')
print(f'\nInformation Criteria:')
print(f'  AIC:  {var_result.aic:.4f}')
print(f'  BIC:  {var_result.bic:.4f}')
print(f'  HQ:   {var_result.hqic:.4f}')
print(f'\nCoefficient matrices shape: {var_result.coefs.shape}')
print(f'  (p={var_result.k_ar} lags x K={var_result.neqs} x K={var_result.neqs})')

### 5b. VECM Estimation

In [ ]:
# Fit VECM model
vecm_rank = max(coint_rank, 1)  # at least 1 for demonstration
vecm_model = VECM(lags=p_opt, coint_rank=vecm_rank, deterministic='ci')
vecm_result = vecm_model.fit(data, names=var_names)

print(f'=== VECM Summary (rank={vecm_rank}) ===')
print(f'Cointegration rank: {vecm_result.coint_rank}')
print(f'Lags (in differences): {vecm_result.k_ar}')

print(f'\nAlpha (adjustment coefficients):')
alpha_df = pd.DataFrame(vecm_result.alpha, index=var_names,
                        columns=[f'EC{i+1}' for i in range(vecm_rank)])
print(alpha_df.round(4).to_string())

print(f'\nBeta (cointegrating vectors):')
beta_df = pd.DataFrame(vecm_result.beta[:K], index=var_names,
                       columns=[f'CV{i+1}' for i in range(vecm_rank)])
print(beta_df.round(4).to_string())

print(f'\nEigenvalues: {np.round(vecm_result.eigenvalues, 4)}')

### 5c. ARDL Bounds Test

In [ ]:
# ARDL bounds test (GDP as dependent, others as regressors)
y_ardl = df['gdp'].values
x_ardl = df[['inflation', 'fed_funds', 'unemployment']].values

ardl_model = ARDL(max_p=4, max_q=4, criterion='aic')
ardl_result = ardl_model.fit(y_ardl, x_ardl)

print('=== ARDL Model ===')
print(f'Dependent variable lags (p): {ardl_result.y_lags}')
print(f'Regressor lags (q):          {ardl_result.x_lags}')
print(f'AIC: {ardl_result.aic:.4f}')
print(f'BIC: {ardl_result.bic:.4f}')
print(f'R-squared: {ardl_result.r_squared:.4f}')

# Bounds test for long-run relationship
bounds = ardl_result.bounds_test()
print(f'\n=== Pesaran Bounds Test ===')
print(f'F-statistic: {bounds["F_statistic"]:.4f}')
print(f'Conclusion:  {bounds["conclusion"]}')

# Long-run coefficients
lr_coefs = ardl_result.long_run_coefficients
print(f'\nLong-run coefficients:')
for i, name in enumerate(['inflation', 'fed_funds', 'unemployment']):
    if i < len(lr_coefs):
        print(f'  {name}: {lr_coefs[i]:.4f}')

---
## 6. Impulse Response Functions (IRF) and FEVD

IRF shows how each variable responds to a one-standard-deviation shock in another variable.
FEVD decomposes the forecast error variance of each variable into contributions from all shocks.

In [ ]:
# Compute IRF with bootstrap confidence intervals
irf_result = var_result.irf(periods=20, method='cholesky', sigs=0.95, runs=500)

print('=== Impulse Response Functions ===')
print(f'Periods: {irf_result.irfs.shape[0]}')
print(f'Shape: {irf_result.irfs.shape} (periods+1 x K x K)')
print(f'Bootstrap runs: 500')

# Show impact effects (period 0)
print(f'\nImpact matrix (period 0):')
impact_df = pd.DataFrame(
    irf_result.irfs[0],
    index=[f'Response: {n}' for n in var_names],
    columns=[f'Shock: {n}' for n in var_names]
)
print(impact_df.round(4).to_string())

In [ ]:
# GRAPH 3: IRF plot
fig = plot_irf(
    irf_results=irf_result,
    var_names=var_names,
    title='Impulse Response Functions (Cholesky)',
    figsize=(14, 12)
)
plt.tight_layout()
plt.show()

In [ ]:
# Compute FEVD
fevd_result = var_result.fevd(periods=20, method='cholesky')

print('=== Forecast Error Variance Decomposition ===')
print(f'Shape: {fevd_result.decomp.shape} (periods+1 x K x K)')

# FEVD at horizon 10
print(f'\nFEVD at horizon 10:')
fevd_h10 = pd.DataFrame(
    fevd_result.decomp[10] * 100,
    index=[f'{n}' for n in var_names],
    columns=[f'Shock: {n}' for n in var_names]
)
print(fevd_h10.round(2).to_string())
print('(Values in % of total forecast error variance)')

In [ ]:
# GRAPH 4: FEVD plot
fig = plot_fevd(
    fevd_results=fevd_result,
    var_names=var_names,
    title='Forecast Error Variance Decomposition',
    figsize=(14, 10)
)
plt.tight_layout()
plt.show()

---
## 7. Granger Causality

Granger causality tests whether past values of one variable help predict another,
beyond the information in its own past values.

In [ ]:
# Test all pairwise Granger causality relations
print('=== Pairwise Granger Causality Tests ===')
print(f'{"Causing":>15} --> {"Caused":<15} {"F-stat":>10} {"p-value":>10} {"Decision":>15}')
print('=' * 70)

granger_matrix = np.zeros((K, K))

for i, caused in enumerate(var_names):
    for j, causing in enumerate(var_names):
        if i == j:
            continue
        gc = granger_causality(var_result, caused=caused, causing=causing)
        granger_matrix[i, j] = gc.pvalue
        decision = 'Granger-causes' if gc.pvalue < 0.05 else 'No causality'
        marker = '*' if gc.pvalue < 0.05 else ' '
        print(f'{causing:>15} --> {caused:<15} {gc.fstat:>10.4f} {gc.pvalue:>10.4f} {decision:>15} {marker}')

print('\n* = significant at 5% level')

In [ ]:
# GRAPH 5: Granger causality heatmap
fig, ax = plt.subplots(figsize=(8, 6))

# Fill diagonal with NaN
gc_display = granger_matrix.copy()
np.fill_diagonal(gc_display, np.nan)

im = ax.imshow(gc_display, cmap='RdYlGn', vmin=0, vmax=0.2, aspect='auto')
ax.set_xticks(range(K))
ax.set_yticks(range(K))
ax.set_xticklabels(var_names, rotation=45, ha='right')
ax.set_yticklabels(var_names)
ax.set_xlabel('Causing variable')
ax.set_ylabel('Caused variable')

for i in range(K):
    for j in range(K):
        if i != j:
            val = granger_matrix[i, j]
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=10, fontweight='bold' if val < 0.05 else 'normal')

plt.colorbar(im, ax=ax, label='p-value')
ax.set_title('Granger Causality p-values (green = significant)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 8. Structural VAR (SVAR)

SVAR identifies structural shocks using economic theory. We use Cholesky decomposition
with the ordering: GDP -> Inflation -> Fed Funds -> Unemployment.

This ordering implies:
- GDP responds only to its own structural shock contemporaneously
- Inflation responds to GDP and its own shock contemporaneously
- Fed Funds responds to GDP, inflation, and its own shock
- Unemployment responds to all shocks contemporaneously

In [ ]:
# SVAR with Cholesky decomposition
svar = SVAR(var_result, method='cholesky')

print('=== Structural VAR (Cholesky) ===')
print(f'Method: {svar.method}')
print(f'Variable ordering: {var_names}')

print(f'\nStructural impact matrix (A0_inv):')
a0_df = pd.DataFrame(
    svar.A0_inv,
    index=var_names,
    columns=[f'Shock_{n}' for n in var_names]
)
print(a0_df.round(4).to_string())

In [ ]:
# Structural IRF
svar_irf = svar.irf(periods=20)

print('=== Structural Impulse Response Functions ===')
print(f'Shape: {svar_irf.irfs.shape}')

In [ ]:
# GRAPH 6: Structural IRF - Selected responses
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
periods_range = np.arange(svar_irf.irfs.shape[0])

# Response of GDP to fed_funds shock
ax = axes[0, 0]
ax.plot(periods_range, svar_irf.irfs[:, 0, 2], 'steelblue', linewidth=1.5)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.fill_between(periods_range, svar_irf.irfs[:, 0, 2], 0, alpha=0.2, color='steelblue')
ax.set_title('GDP <- Fed Funds Shock', fontsize=11)
ax.grid(True, alpha=0.3)

# Response of inflation to fed_funds shock
ax = axes[0, 1]
ax.plot(periods_range, svar_irf.irfs[:, 1, 2], 'darkorange', linewidth=1.5)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.fill_between(periods_range, svar_irf.irfs[:, 1, 2], 0, alpha=0.2, color='darkorange')
ax.set_title('Inflation <- Fed Funds Shock', fontsize=11)
ax.grid(True, alpha=0.3)

# Response of unemployment to GDP shock
ax = axes[1, 0]
ax.plot(periods_range, svar_irf.irfs[:, 3, 0], 'red', linewidth=1.5)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.fill_between(periods_range, svar_irf.irfs[:, 3, 0], 0, alpha=0.2, color='red')
ax.set_title('Unemployment <- GDP Shock', fontsize=11)
ax.grid(True, alpha=0.3)

# Response of fed_funds to inflation shock
ax = axes[1, 1]
ax.plot(periods_range, svar_irf.irfs[:, 2, 1], 'green', linewidth=1.5)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.fill_between(periods_range, svar_irf.irfs[:, 2, 1], 0, alpha=0.2, color='green')
ax.set_title('Fed Funds <- Inflation Shock', fontsize=11)
ax.grid(True, alpha=0.3)

plt.suptitle('Structural Impulse Responses (SVAR Cholesky)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Spillover Index (Diebold-Yilmaz)

The spillover index measures the degree of interconnectedness between variables
using the generalized forecast error variance decomposition framework.

In [ ]:
# Compute spillover index
spillover = SpilloverIndex(lags=p_opt, horizon=10)
spill_result = spillover.fit(data)

print('=== Diebold-Yilmaz Spillover Index ===')
print(f'Total Spillover Index: {spill_result.total_spillover:.2f}%')

print(f'\nSpillover Table:')
spill_df = pd.DataFrame(
    spill_result.spillover_table,
    index=var_names,
    columns=var_names
)
print(spill_df.round(2).to_string())

print(f'\nDirectional spillovers FROM others:')
for i, name in enumerate(var_names):
    from_others = spill_result.from_spillover[i]
    print(f'  {name}: {from_others:.2f}%')

print(f'\nDirectional spillovers TO others:')
for i, name in enumerate(var_names):
    to_others = spill_result.to_spillover[i]
    print(f'  {name}: {to_others:.2f}%')

In [ ]:
# GRAPH 7: Spillover heatmap
fig = plot_heatmap(
    spillover=spill_result,
    var_names=var_names,
    title='Spillover Table (Diebold-Yilmaz)',
    figsize=(8, 6)
)
plt.tight_layout()
plt.show()

In [ ]:
# GRAPH 8: Spillover network
fig = plot_network(
    spillover=spill_result,
    var_names=var_names,
    threshold=2.0,
    title='Spillover Network',
    figsize=(8, 8)
)
plt.tight_layout()
plt.show()

---
## 10. Multivariate Forecasting

In [ ]:
# VAR forecast
h = 8  # 8 quarters = 2 years
fc_var = var_result.forecast(steps=h)

# Create forecast DataFrame
fc_dates = pd.date_range(
    start=df.index[-1] + pd.DateOffset(months=3),
    periods=h, freq='QS'
)
fc_df = pd.DataFrame(fc_var, index=fc_dates, columns=var_names)

print('=== VAR Forecast (8 quarters ahead) ===')
print(fc_df.round(4).to_string())

In [ ]:
# GRAPH 9: Multivariate forecast plot
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

n_hist = 24  # show last 24 quarters of history

for ax, col, title, color in zip(axes.flat, var_names, titles, colors):
    # Historical data
    hist_idx = df.index[-n_hist:]
    ax.plot(hist_idx, df[col].values[-n_hist:], color=color,
            linewidth=1.5, label='Historical')
    
    # Forecast
    ax.plot(fc_dates, fc_df[col], color=color, linewidth=1.5,
            linestyle='--', marker='o', markersize=3, label='Forecast')
    
    # Connect history to forecast
    ax.plot([hist_idx[-1], fc_dates[0]],
            [df[col].values[-1], fc_df[col].iloc[0]],
            color=color, linestyle=':', alpha=0.5)
    
    ax.axvline(x=hist_idx[-1], color='gray', linestyle=':', alpha=0.5)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('VAR Multivariate Forecast', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 11. Integration Between Modules

This section demonstrates how chronobox modules chain together seamlessly.

In [ ]:
# Complete pipeline: Tests -> VAR -> SVAR -> IRF -> Spillover
# All in a few lines, chaining chronobox functions

# 1. Test stationarity for each variable
from chronobox.tests_stat import adf_test
stationary_flags = {col: adf_test(df[col].values).reject for col in var_names}
print('Stationarity:', stationary_flags)

# 2. Fit VAR
from chronobox import VAR
var = VAR(lags=2, trend='c')
res = var.fit(data, names=var_names)
print(f'VAR({res.k_ar}) fitted, stable={res.is_stable}')

# 3. Structural analysis
from chronobox import SVAR
svar = SVAR(res, method='cholesky')
irf = svar.irf(periods=12)
print(f'SVAR IRF computed: {irf.irfs.shape}')

# 4. Spillover analysis
from chronobox.analysis import SpilloverIndex
si = SpilloverIndex(lags=2, horizon=10)
sp = si.fit(data)
print(f'Total spillover: {sp.total_spillover:.1f}%')

# 5. Granger causality
from chronobox.analysis import granger_causality
gc = granger_causality(res, caused='unemployment', causing='gdp')
print(f'GDP Granger-causes unemployment: p={gc.pvalue:.4f}')

---
## Exercicio

**Extend this analysis with a Bayesian VAR (BVAR)**

Bayesian VAR uses prior distributions on the coefficients to regularize estimation,
which is particularly useful when the number of parameters is large relative to observations.

Tasks:
1. Fit a `BayesianVAR` with Minnesota prior using `lags=2`
2. Compare the BVAR forecast (8 quarters) with the VAR forecast above
3. Plot both forecasts side by side for each variable
4. Which model produces tighter prediction intervals? Why?
5. Try different priors ('minnesota', 'normal_wishart') and compare

In [ ]:
# === YOUR CODE HERE ===

# 1. Fit Bayesian VAR
# from chronobox import BayesianVAR
#
# bvar = BayesianVAR(lags=2, prior='minnesota')
# bvar_result = bvar.fit(data, n_draws=5000, burnin=1000, seed=42)
#
# print('BVAR fitted')
# print(f'Log marginal likelihood: {bvar_result.log_marginal_likelihood:.4f}')

# 2. Forecast
# bvar_fc = bvar_result.forecast(steps=8)
# print('BVAR forecast mean:')
# print(bvar_fc['mean'])

# 3. Compare with VAR forecast
# fig, axes = plt.subplots(2, 2, figsize=(14, 8))
# for ax, col_idx, name in zip(axes.flat, range(4), var_names):
#     ax.plot(fc_var[:, col_idx], 'b-o', label='VAR', markersize=3)
#     ax.plot(bvar_fc['mean'][:, col_idx], 'r-s', label='BVAR', markersize=3)
#     ax.fill_between(range(8),
#                     bvar_fc['lower_95'][:, col_idx],
#                     bvar_fc['upper_95'][:, col_idx],
#                     alpha=0.2, color='red', label='BVAR 95% CI')
#     ax.set_title(name)
#     ax.legend(fontsize=8)
#     ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

# 4 & 5. Experiment with different priors
# bvar_nw = BayesianVAR(lags=2, prior='normal_wishart')
# bvar_nw_result = bvar_nw.fit(data, n_draws=5000, burnin=1000, seed=42)

print('Complete the exercise above!')